# Innere Staerke - Video Factory (SadTalker)
**Natural talking head animation + German TTS + Captions**

Run all cells top to bottom. Upload avatar, enter script, get video!

In [ ]:
#@title 1. Clone Pipeline Repo + Install SadTalker (Run once ~3 min)
import os

# Clone our pipeline repo
if not os.path.exists('/content/PILELINE'):
    !git clone https://github.com/kaleem78786/PILELINE.git /content/PILELINE
    print('Pipeline repo cloned!')
else:
    os.chdir('/content/PILELINE')
    !git pull
    print('Pipeline repo updated!')

# Install edge-tts
!pip install edge-tts pyngrok flask flask-cors -q

# Clone & install SadTalker
if not os.path.exists('/content/SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker
    os.chdir('/content/SadTalker')
    !pip install -r requirements.txt -q
    !pip install dlib-bin -q
else:
    print('SadTalker already installed!')

# Download SadTalker models
os.chdir('/content/SadTalker')
if not os.path.exists('/content/SadTalker/checkpoints/SadTalker_V0.0.2_512.safetensors'):
    !bash scripts/download_models.sh
else:
    print('Models already downloaded!')

!mkdir -p /content/workspace /content/workspace/output
print('\nDone! Installation complete.')

In [ ]:
#@title 2. Upload Avatar Image
from google.colab import files
import shutil
from PIL import Image
from IPython.display import display

# Check if avatar already exists from repo
repo_avatar = '/content/PILELINE/avatars/avatar.png'
workspace_avatar = '/content/workspace/avatar.png'

if os.path.exists(repo_avatar):
    shutil.copy(repo_avatar, workspace_avatar)
    print('Avatar loaded from repo!')
else:
    print('Upload your avatar image (transparent PNG)...')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.move(fname, workspace_avatar)

img = Image.open(workspace_avatar)
print(f'Avatar: {img.size}, Mode: {img.mode}')
img_show = img.copy()
img_show.thumbnail((300, 300))
display(img_show)

avatar_path = workspace_avatar

In [ ]:
#@title 3. Generate Video
#@markdown ### German Script:
german_script = "Ich werde mit etwas beginnen, das dich vielleicht ein wenig unwohl fuehlen laesst. Wenn du merkst, dass du dich defensiv fuehlst, ist das voellig in Ordnung. Bemerke es einfach. Aber frage dich warum. Denn ich garantiere dir, wenn du bei mir bleibst, wird dir das wahrscheinlich jede einzelne Beziehung erklaeren, die du jemals hattest." #@param {type:"string"}
#@markdown ### Voice:
voice = "de-DE-SeraphinaMultilingualNeural" #@param ["de-DE-SeraphinaMultilingualNeural", "de-DE-AmalaNeural", "de-DE-KatjaNeural", "de-DE-ConradNeural", "de-DE-KillianNeural"]
#@markdown ### Settings:
enhancer = "gfpgan" #@param ["gfpgan", "RestoreFormer", "none"]
still_mode = True #@param {type:"boolean"}
preprocess = "full" #@param ["crop", "resize", "full"]
expression_scale = 1.2 #@param {type:"slider", min:0.5, max:2.0, step:0.1}

import asyncio, edge_tts, subprocess, time, glob

audio_path = '/content/workspace/audio.mp3'
srt_path = '/content/workspace/subtitles.srt'

# --- TTS ---
print('Step 1/3: Generating German TTS audio...')
async def gen_tts():
    comm = edge_tts.Communicate(german_script, voice=voice)
    sub = edge_tts.SubMaker()
    with open(audio_path, 'wb') as f:
        async for c in comm.stream():
            if c['type'] == 'audio': f.write(c['data'])
            elif c['type'] in ('WordBoundary','SentenceBoundary'): sub.feed(c)
    srt = sub.get_srt()
    with open(srt_path, 'w') as f: f.write(srt)
    return srt
srt_content = await gen_tts()
blocks = [b for b in srt_content.strip().split('\n\n') if b.strip()]
print(f'  Audio: OK | Subtitles: {len(blocks)} blocks')

# --- SadTalker ---
print('\nStep 2/3: SadTalker lip-sync animation...')
os.chdir('/content/SadTalker')

cmd = ['python', 'inference.py',
    '--driven_audio', audio_path,
    '--source_image', avatar_path,
    '--result_dir', '/content/workspace/output',
    '--preprocess', preprocess,
    '--size', '512',
    '--expression_scale', str(expression_scale),
]
if still_mode: cmd.append('--still')
if enhancer != 'none': cmd.extend(['--enhancer', enhancer])

t0 = time.time()
res = subprocess.run(cmd, capture_output=True, text=True)
print(f'  SadTalker done in {time.time()-t0:.0f}s')
if res.returncode != 0:
    print(f'  Error: {res.stderr[-300:]}')

# Find latest mp4
vids = sorted(glob.glob('/content/workspace/output/**/*.mp4', recursive=True), key=os.path.getmtime, reverse=True)
sadtalker_video = vids[0] if vids else None
print(f'  Video: {sadtalker_video}')

# --- Compose Final ---
print('\nStep 3/3: Composing final video (black BG + character + captions)...')
final_output = '/content/workspace/final_video.mp4'

if sadtalker_video:
    # Build caption filter with 3 colors (yellow/red/white rotating)
    filter_complex = (
        f"[0:v]scale=540:-1[avatar];"
        f"color=c=black:s=1920x1080:d=9999[bg];"
        f"[bg][avatar]overlay=80:(H-h)/2:shortest=1[base];"
        f"[base]subtitles='/content/workspace/subtitles.srt':force_style='"
        f"FontName=DejaVu Sans,FontSize=30,Bold=1,"
        f"PrimaryColour=&H0000FFFF,"
        f"OutlineColour=&H00000000,BackColour=&H80000000,"
        f"Outline=2,Shadow=1,"
        f"MarginL=550,MarginR=40,MarginV=50,Alignment=6'[v]"
    )
    comp = subprocess.run([
        'ffmpeg', '-y', '-i', sadtalker_video, '-i', audio_path,
        '-filter_complex', filter_complex,
        '-map', '[v]', '-map', '1:a',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '20',
        '-c:a', 'aac', '-b:a', '192k',
        '-shortest', '-pix_fmt', 'yuv420p', '-movflags', '+faststart',
        final_output
    ], capture_output=True, text=True)
    
    if comp.returncode != 0:
        print('  Subtitle embed failed, composing without...')
        subprocess.run([
            'ffmpeg', '-y', '-i', sadtalker_video, '-i', audio_path,
            '-filter_complex',
            f"[0:v]scale=540:-1[avatar];"
            f"color=c=black:s=1920x1080:d=9999[bg];"
            f"[bg][avatar]overlay=80:(H-h)/2:shortest=1[v]",
            '-map', '[v]', '-map', '1:a',
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '20',
            '-c:a', 'aac', '-b:a', '192k',
            '-shortest', '-pix_fmt', 'yuv420p', final_output
        ], capture_output=True, text=True)

if os.path.exists(final_output):
    mb = os.path.getsize(final_output)/(1024*1024)
    print(f'\nFINAL VIDEO: {final_output} ({mb:.1f} MB)')
else:
    print('Video generation failed!')

In [ ]:
#@title 4. Preview and Download
from IPython.display import HTML, display
from google.colab import files
import base64

final_output = '/content/workspace/final_video.mp4'

if os.path.exists(final_output):
    with open(final_output, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'<video width=960 controls><source src="data:video/mp4;base64,{b64}"></video>'))
    print('Downloading...')
    files.download(final_output)
else:
    # Try SadTalker raw output
    import glob
    vids = sorted(glob.glob('/content/workspace/output/**/*.mp4', recursive=True), key=os.path.getmtime, reverse=True)
    if vids:
        print(f'Using raw SadTalker output: {vids[0]}')
        files.download(vids[0])
    else:
        print('No video found! Run Step 3 first.')

In [ ]:
#@title 5. (Optional) Start API Server for Local Automation
#@markdown Ngrok auth token (get from https://ngrok.com):
ngrok_token = "" #@param {type:"string"}

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading, base64, asyncio, glob

app = Flask(__name__)
CORS(app)

@app.route('/health')
def health():
    import torch
    return jsonify({'status':'ok','gpu':torch.cuda.is_available()})

@app.route('/generate', methods=['POST'])
def api_generate():
    data = request.json
    script = data.get('script','')
    voice_name = data.get('voice','de-DE-SeraphinaMultilingualNeural')
    
    if 'avatar_base64' in data:
        img_bytes = base64.b64decode(data['avatar_base64'])
        with open('/content/workspace/api_avatar.png','wb') as f: f.write(img_bytes)
        src = '/content/workspace/api_avatar.png'
    else:
        src = avatar_path
    
    try:
        # TTS
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        async def do_tts():
            c = edge_tts.Communicate(script, voice=voice_name)
            s = edge_tts.SubMaker()
            with open('/content/workspace/api_audio.mp3','wb') as f:
                async for ch in c.stream():
                    if ch['type']=='audio': f.write(ch['data'])
                    elif ch['type'] in ('WordBoundary','SentenceBoundary'): s.feed(ch)
            with open('/content/workspace/api_subs.srt','w') as f: f.write(s.get_srt())
        loop.run_until_complete(do_tts())
        
        # SadTalker
        os.chdir('/content/SadTalker')
        subprocess.run(['python','inference.py',
            '--driven_audio','/content/workspace/api_audio.mp3',
            '--source_image',src,
            '--result_dir','/content/workspace/api_out',
            '--still','--preprocess','full',
            '--enhancer','gfpgan','--size','512',
        ], capture_output=True, text=True, timeout=600)
        
        vids = sorted(glob.glob('/content/workspace/api_out/**/*.mp4',recursive=True), key=os.path.getmtime, reverse=True)
        if vids:
            with open(vids[0],'rb') as f: v64 = base64.b64encode(f.read()).decode()
            return jsonify({'status':'ok','video_base64':v64})
        return jsonify({'status':'error','msg':'No video'}),500
    except Exception as e:
        return jsonify({'status':'error','msg':str(e)}),500

if ngrok_token: ngrok.set_auth_token(ngrok_token)
url = ngrok.connect(5000)
print(f'\nAPI Server: {url}')
print(f'\nLocal command:')
print(f'  python3 colab_client.py --url "{url}" --batch scripts/')
threading.Thread(target=lambda: app.run(port=5000)).start()